In [1]:
# ============================================================
# CELL 1: Install and Imports
# ============================================================

!pip install -q librosa tqdm

import os
import glob
import re
import pickle
import random
import warnings

import numpy as np
import librosa

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import train_test_split
from tqdm import tqdm

from collections import Counter

warnings.filterwarnings("ignore")

print("Imports completed successfully!")

Imports completed successfully!


# CELL 2: Global Configuration

In [2]:

CONFIG = {
    "DATA_DIR": "/kaggle/input/datasets/kaggli123/audio-dataset/audio",
    "OUTPUT_DIR": "/kaggle/working/processed",

    "EMOTIONS": {
        "happy": 0,
        "sad": 1,
        "angry": 2,
        "fear": 3,
        "surprise": 4
    },

    "SAMPLE_RATE": 22050,
    "DURATION": 3,
    "N_MFCC": 40,
    "N_FFT": 2048,
    "HOP_LENGTH": 512,
    "MAX_LEN": 130,

    "RANDOM_SEED": 42
}

random.seed(CONFIG["RANDOM_SEED"])
np.random.seed(CONFIG["RANDOM_SEED"])
torch.manual_seed(CONFIG["RANDOM_SEED"])

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG["RANDOM_SEED"])

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)

print("Device:", DEVICE)
print("Data directory:", CONFIG["DATA_DIR"])
print("Output directory:", CONFIG["OUTPUT_DIR"])
print("Target emotions:", CONFIG["EMOTIONS"])
print("Configuration completed successfully!")

Device: cuda
Data directory: /kaggle/input/datasets/kaggli123/audio-dataset/audio
Output directory: /kaggle/working/processed
Target emotions: {'happy': 0, 'sad': 1, 'angry': 2, 'fear': 3, 'surprise': 4}
Configuration completed successfully!


# CELL 3: File Parser

In [3]:

def parse_files(data_dir, config):
    """
    Parse audio files from the target emotion folders.

    Detects whether each file belongs to RAVDESS or CREMA-D,
    extracts its emotion label from the filename, validates the
    folder label, and returns valid file paths with integer labels.
    """

    file_list = []

    ravdess_emotions = {
        "03": "happy",
        "04": "sad",
        "05": "angry",
        "06": "fear",
        "08": "surprise"
    }

    crema_emotions = {
        "HAP": "happy",
        "SAD": "sad",
        "ANG": "angry",
        "FEA": "fear"
    }

    ravdess_count = 0
    crema_count = 0
    warning_count = 0

    for folder_name in config["EMOTIONS"].keys():

        folder_path = os.path.join(data_dir, folder_name)

        if not os.path.isdir(folder_path):
            print(f"Warning: Folder not found: {folder_path}")
            continue

        wav_files = glob.glob(os.path.join(folder_path, "*.wav"))

        for filepath in wav_files:

            filename = os.path.basename(filepath)
            detected_emotion = None
            dataset_type = None

            # ------------------------------------------------
            # RAVDESS
            # Example: 03-01-05-01-01-01-01.wav
            # ------------------------------------------------
            if re.match(r"^\d{2}-", filename):

                parts = filename.replace(".wav", "").split("-")

                if len(parts) >= 3:
                    emotion_code = parts[2]

                    if emotion_code in ravdess_emotions:
                        detected_emotion = ravdess_emotions[emotion_code]
                        dataset_type = "RAVDESS"

            # ------------------------------------------------
            # CREMA-D
            # Example: 1001_DFA_ANG_XX.wav
            # ------------------------------------------------
            elif "_" in filename:

                parts = filename.replace(".wav", "").split("_")

                if len(parts) >= 3:
                    emotion_code = parts[2]

                    if emotion_code in crema_emotions:
                        detected_emotion = crema_emotions[emotion_code]
                        dataset_type = "CREMA-D"

            # ------------------------------------------------
            # Skip invalid or unwanted files
            # ------------------------------------------------
            if detected_emotion is None:
                continue

            # ------------------------------------------------
            # Validate folder name against filename emotion
            # ------------------------------------------------
            if folder_name != detected_emotion:

                warning_count += 1

                print(
                    f"Warning: Folder mismatch | "
                    f"Folder: {folder_name} | "
                    f"Detected: {detected_emotion} | "
                    f"File: {filename}"
                )

                continue

            label = config["EMOTIONS"][detected_emotion]

            file_list.append((filepath, label))

            if dataset_type == "RAVDESS":
                ravdess_count += 1

            elif dataset_type == "CREMA-D":
                crema_count += 1

    # --------------------------------------------------------
    # Class distribution
    # --------------------------------------------------------

    label_counts = Counter(label for _, label in file_list)

    class_distribution = {
        emotion: label_counts.get(label, 0)
        for emotion, label in config["EMOTIONS"].items()
    }

    print()
    print("=" * 60)
    print("FILE PARSING RESULTS")
    print("=" * 60)

    print(f"Total valid files found: {len(file_list)}")
    print(f"Class distribution: {class_distribution}")
    print(f"Files from RAVDESS: {ravdess_count}")
    print(f"Files from CREMA-D: {crema_count}")
    print(f"Folder mismatch warnings: {warning_count}")

    print("=" * 60)

    return file_list


print("File parser function created successfully!")

File parser function created successfully!


In [4]:
# ============================================================
# CELL 4: MFCC Feature Extraction
# ============================================================

def extract_mfcc(filepath, config):
    """
    Load an audio file and extract a normalized MFCC feature matrix.

    The audio is resampled to the configured sample rate, padded
    or trimmed to the configured duration, converted into MFCC
    features, normalized, and padded or truncated to MAX_LEN.
    """

    audio, sr = librosa.load(
        filepath,
        sr=config["SAMPLE_RATE"]
    )

    target_length = config["SAMPLE_RATE"]*config["DURATION"]

    if len(audio)<target_length:
        audio = np.pad(
            audio,
            (0,target_length-len(audio)),
            mode="constant"
        )
    else:
        audio = audio[:target_length]

    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=config["SAMPLE_RATE"],
        n_mfcc=config["N_MFCC"],
        n_fft=config["N_FFT"],
        hop_length=config["HOP_LENGTH"]
    )

    mean = np.mean(mfcc)
    std = np.std(mfcc)

    mfcc = (mfcc-mean)/(std+1e-8)

    if mfcc.shape[1]<config["MAX_LEN"]:
        mfcc = np.pad(
            mfcc,
            (
                (0,0),
                (0,config["MAX_LEN"]-mfcc.shape[1])
            ),
            mode="constant"
        )
    else:
        mfcc = mfcc[:,:config["MAX_LEN"]]

    return mfcc.astype(np.float32)


def extract_and_save_all(file_list, config):
    """
    Extract MFCC features from all valid audio files and save
    the features, labels, and original file paths to disk.
    """

    features = []
    labels = []
    valid_filepaths = []

    skipped_files = 0

    print("Starting MFCC feature extraction...")
    print(f"Total files to process: {len(file_list)}")
    print()

    for filepath, label in tqdm(
        file_list,
        desc="Extracting MFCCs"
    ):

        try:
            mfcc = extract_mfcc(filepath, config)

            if mfcc.shape!=(config["N_MFCC"],config["MAX_LEN"]):
                print(
                    f"Skipping unexpected shape: "
                    f"{filepath} -> {mfcc.shape}"
                )
                skipped_files += 1
                continue

            features.append(mfcc)
            labels.append(label)
            valid_filepaths.append(filepath)

        except Exception as e:
            print(f"\nError processing: {filepath}")
            print(f"Error: {e}")
            skipped_files += 1

    features = np.array(features,dtype=np.float32)
    labels = np.array(labels,dtype=np.int64)

    features_path = os.path.join(
        config["OUTPUT_DIR"],
        "features.npy"
    )

    labels_path = os.path.join(
        config["OUTPUT_DIR"],
        "labels.npy"
    )

    filepaths_path = os.path.join(
        config["OUTPUT_DIR"],
        "filepaths.pkl"
    )

    np.save(features_path,features)
    np.save(labels_path,labels)

    with open(filepaths_path,"wb") as f:
        pickle.dump(valid_filepaths,f)

    print()
    print("="*60)
    print("MFCC EXTRACTION RESULTS")
    print("="*60)
    print(f"Features shape: {features.shape}")
    print(f"Labels shape: {labels.shape}")
    print(f"Valid files: {len(features)}")
    print(f"Skipped files: {skipped_files}")
    print()
    print(f"Saved: {features_path}")
    print(f"Saved: {labels_path}")
    print(f"Saved: {filepaths_path}")
    print("="*60)

    return features,labels,valid_filepaths


print("MFCC extraction functions created successfully!")

MFCC extraction functions created successfully!


# ============================================================
# CELL 4: MFCC Feature Extraction
# ============================================================


In [5]:

def extract_mfcc(filepath, config):
    """
    Load an audio file and extract a normalized MFCC feature matrix.

    The audio is resampled to the configured sample rate, padded
    or trimmed to the configured duration, converted into MFCC
    features, normalized, and padded or truncated to MAX_LEN.
    """

    audio, sr = librosa.load(
        filepath,
        sr=config["SAMPLE_RATE"]
    )

    target_length = config["SAMPLE_RATE"]*config["DURATION"]

    if len(audio)<target_length:
        audio = np.pad(
            audio,
            (0,target_length-len(audio)),
            mode="constant"
        )
    else:
        audio = audio[:target_length]

    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=config["SAMPLE_RATE"],
        n_mfcc=config["N_MFCC"],
        n_fft=config["N_FFT"],
        hop_length=config["HOP_LENGTH"]
    )

    mean = np.mean(mfcc)
    std = np.std(mfcc)

    mfcc = (mfcc-mean)/(std+1e-8)

    if mfcc.shape[1]<config["MAX_LEN"]:
        mfcc = np.pad(
            mfcc,
            (
                (0,0),
                (0,config["MAX_LEN"]-mfcc.shape[1])
            ),
            mode="constant"
        )
    else:
        mfcc = mfcc[:,:config["MAX_LEN"]]

    return mfcc.astype(np.float32)


def extract_and_save_all(file_list, config):
    """
    Extract MFCC features from all valid audio files and save
    the features, labels, and original file paths to disk.
    """

    features = []
    labels = []
    valid_filepaths = []

    skipped_files = 0

    print("Starting MFCC feature extraction...")
    print(f"Total files to process: {len(file_list)}")
    print()

    for filepath, label in tqdm(
        file_list,
        desc="Extracting MFCCs"
    ):

        try:
            mfcc = extract_mfcc(filepath, config)

            if mfcc.shape!=(config["N_MFCC"],config["MAX_LEN"]):
                print(
                    f"Skipping unexpected shape: "
                    f"{filepath} -> {mfcc.shape}"
                )
                skipped_files += 1
                continue

            features.append(mfcc)
            labels.append(label)
            valid_filepaths.append(filepath)

        except Exception as e:
            print(f"\nError processing: {filepath}")
            print(f"Error: {e}")
            skipped_files += 1

    features = np.array(features,dtype=np.float32)
    labels = np.array(labels,dtype=np.int64)

    features_path = os.path.join(
        config["OUTPUT_DIR"],
        "features.npy"
    )

    labels_path = os.path.join(
        config["OUTPUT_DIR"],
        "labels.npy"
    )

    filepaths_path = os.path.join(
        config["OUTPUT_DIR"],
        "filepaths.pkl"
    )

    np.save(features_path,features)
    np.save(labels_path,labels)

    with open(filepaths_path,"wb") as f:
        pickle.dump(valid_filepaths,f)

    print()
    print("="*60)
    print("MFCC EXTRACTION RESULTS")
    print("="*60)
    print(f"Features shape: {features.shape}")
    print(f"Labels shape: {labels.shape}")
    print(f"Valid files: {len(features)}")
    print(f"Skipped files: {skipped_files}")
    print()
    print(f"Saved: {features_path}")
    print(f"Saved: {labels_path}")
    print(f"Saved: {filepaths_path}")
    print("="*60)

    return features,labels,valid_filepaths


print("MFCC extraction functions created successfully!")

MFCC extraction functions created successfully!


# CELL 5: Run Feature Extraction

In [6]:

file_list = parse_files(
    CONFIG["DATA_DIR"],
    CONFIG
)

print()
print(f"Files ready for extraction: {len(file_list)}")

features, labels, valid_filepaths = extract_and_save_all(
    file_list,
    CONFIG
)

print()
print("Feature extraction complete!")
print(f"Final feature shape: {features.shape}")
print(f"Final label shape: {labels.shape}")


FILE PARSING RESULTS
Total valid files found: 6780
Class distribution: {'happy': 1647, 'sad': 1647, 'angry': 1647, 'fear': 1647, 'surprise': 192}
Files from RAVDESS: 1696
Files from CREMA-D: 5084
Folder mismatch warnings: 0

Files ready for extraction: 6780
Starting MFCC feature extraction...
Total files to process: 6780



Extracting MFCCs: 100%|██████████| 6780/6780 [03:27<00:00, 32.70it/s]



MFCC EXTRACTION RESULTS
Features shape: (6780, 40, 130)
Labels shape: (6780,)
Valid files: 6780
Skipped files: 0

Saved: /kaggle/working/processed/features.npy
Saved: /kaggle/working/processed/labels.npy
Saved: /kaggle/working/processed/filepaths.pkl

Feature extraction complete!
Final feature shape: (6780, 40, 130)
Final label shape: (6780,)


In [7]:
# ============================================================
# DIAGNOSTIC: Find Actual Dataset Path
# ============================================================

print("Contents of /kaggle/input:")
print(os.listdir("/kaggle/input"))

print()
print("Contents of audio_dataset:")
print(os.listdir("/kaggle/input/audio_dataset"))

print()
print("Contents of audio folder:")
print(os.listdir("/kaggle/input/audio_dataset/audio"))

Contents of /kaggle/input:
['datasets']

Contents of audio_dataset:


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/audio_dataset'

In [ ]:
# ============================================================
# DIAGNOSTIC: Find Dataset Location
# ============================================================

for root,dirs,files in os.walk("/kaggle/input/datasets"):
    print(root)

    if root.count(os.sep)>=5:
        dirs.clear()

# ============================================================
# CELL 6: Train / Validation / Test Split
# ============================================================


In [ ]:

features_path = os.path.join(
    CONFIG["OUTPUT_DIR"],
    "features.npy"
)

labels_path = os.path.join(
    CONFIG["OUTPUT_DIR"],
    "labels.npy"
)

X = np.load(features_path)
y = np.load(labels_path)

print("Loaded features:", X.shape)
print("Loaded labels:", y.shape)

# ------------------------------------------------------------
# First split: 85% train+validation, 15% test
# ------------------------------------------------------------

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=CONFIG["RANDOM_SEED"],
    stratify=y
)

# ------------------------------------------------------------
# Second split:
# 70% train and 15% validation of the original dataset
# ------------------------------------------------------------

val_ratio = 0.15/0.85

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=val_ratio,
    random_state=CONFIG["RANDOM_SEED"],
    stratify=y_train_val
)

# ------------------------------------------------------------
# Save split files
# ------------------------------------------------------------

np.save(
    os.path.join(CONFIG["OUTPUT_DIR"],"X_train.npy"),
    X_train
)

np.save(
    os.path.join(CONFIG["OUTPUT_DIR"],"y_train.npy"),
    y_train
)

np.save(
    os.path.join(CONFIG["OUTPUT_DIR"],"X_val.npy"),
    X_val
)

np.save(
    os.path.join(CONFIG["OUTPUT_DIR"],"y_val.npy"),
    y_val
)

np.save(
    os.path.join(CONFIG["OUTPUT_DIR"],"X_test.npy"),
    X_test
)

np.save(
    os.path.join(CONFIG["OUTPUT_DIR"],"y_test.npy"),
    y_test
)

# ------------------------------------------------------------
# Helper function for class distribution
# ------------------------------------------------------------

def get_class_distribution(labels, config):
    """
    Return the number of samples belonging to each emotion class.
    """

    counts = Counter(labels)

    return {
        emotion: counts.get(label,0)
        for emotion,label in config["EMOTIONS"].items()
    }


# ------------------------------------------------------------
# Print split information
# ------------------------------------------------------------

print()
print("="*60)
print("DATASET SPLIT RESULTS")
print("="*60)

print(f"Train features: {X_train.shape}")
print(f"Train labels:   {y_train.shape}")
print(f"Train classes:  {get_class_distribution(y_train,CONFIG)}")

print()

print(f"Validation features: {X_val.shape}")
print(f"Validation labels:   {y_val.shape}")
print(f"Validation classes:  {get_class_distribution(y_val,CONFIG)}")

print()

print(f"Test features: {X_test.shape}")
print(f"Test labels:   {y_test.shape}")
print(f"Test classes:  {get_class_distribution(y_test,CONFIG)}")

print()

print("Saved files:")
print(os.path.join(CONFIG["OUTPUT_DIR"],"X_train.npy"))
print(os.path.join(CONFIG["OUTPUT_DIR"],"y_train.npy"))
print(os.path.join(CONFIG["OUTPUT_DIR"],"X_val.npy"))
print(os.path.join(CONFIG["OUTPUT_DIR"],"y_val.npy"))
print(os.path.join(CONFIG["OUTPUT_DIR"],"X_test.npy"))
print(os.path.join(CONFIG["OUTPUT_DIR"],"y_test.npy"))

print("="*60)
print("Train/Val/Test split completed successfully!")